In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np

In [11]:
torch.manual_seed(1234)
torch.cuda.manual_seed(1234)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [12]:
class TextDataset(Dataset):
    def __init__(self, data, labels, vocab_size=50, seq_length=10):
        self.data = data
        self.labels = labels
        self.vocab_size = vocab_size
        self.seq_length = seq_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        sample = self.data[idx]
        label = self.labels[idx]
        # Convert text to indices (for simplicity, each word is a unique index here)
        sample_idx = [ord(char) % self.vocab_size for char in sample]
        # Pad sequences
        if len(sample_idx) < self.seq_length:
            sample_idx += [0] * (self.seq_length - len(sample_idx))
        else:
            sample_idx = sample_idx[:self.seq_length]
        return torch.tensor(sample_idx), torch.tensor(label)

# Sample data
texts = ["hello world", "how are you", "goodbye now", "see you soon", "hi there"]
labels = [0, 0, 1, 1, 0]  # 0: Greeting, 1: Farewell
dataset = TextDataset(texts, labels)

# Create a DataLoader
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)


In [13]:
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

In [16]:
next(iter(dataloader))

[tensor([[ 3, 11, 11,  0, 48, 21,  1, 32, 10, 11],
         [ 4,  5, 32, 16,  4,  1, 14,  1,  0,  0]]),
 tensor([1, 0])]

In [19]:
class RNNModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(RNNModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        embedded = self.embedding(x)
        rnn_out, hidden = self.rnn(embedded)
        # Take the hidden state of the last RNN cell
        out = self.fc(hidden[-1])
        return out


In [20]:
# Model parameters
vocab_size = 50
embedding_dim = 10
hidden_dim = 20
output_dim = 2

# Instantiate the model
model = RNNModel(vocab_size, embedding_dim, hidden_dim, output_dim)
model

RNNModel(
  (embedding): Embedding(50, 10)
  (rnn): RNN(10, 20, batch_first=True)
  (fc): Linear(in_features=20, out_features=2, bias=True)
)

In [21]:
for p in model.parameters():
  print(type(p), p.size())
  # print(p.numel())

<class 'torch.nn.parameter.Parameter'> torch.Size([50, 10])
<class 'torch.nn.parameter.Parameter'> torch.Size([20, 10])
<class 'torch.nn.parameter.Parameter'> torch.Size([20, 20])
<class 'torch.nn.parameter.Parameter'> torch.Size([20])
<class 'torch.nn.parameter.Parameter'> torch.Size([20])
<class 'torch.nn.parameter.Parameter'> torch.Size([2, 20])
<class 'torch.nn.parameter.Parameter'> torch.Size([2])


In [ ]:
# help(model)

In [23]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

criterion.to(device)
model.to(device)

RNNModel(
  (embedding): Embedding(50, 10)
  (rnn): RNN(10, 20, batch_first=True)
  (fc): Linear(in_features=20, out_features=2, bias=True)
)

In [24]:
epochs = 10

for epoch in range(epochs):
    total_loss = 0.0
    model.train()
    for inputs, labels in dataloader:
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        # Accumulate the loss
        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {total_loss/len(dataloader):.4f}")


Epoch [1/10], Loss: 0.7377
Epoch [2/10], Loss: 0.7008
Epoch [3/10], Loss: 0.6280
Epoch [4/10], Loss: 0.5982
Epoch [5/10], Loss: 0.5781
Epoch [6/10], Loss: 0.5536
Epoch [7/10], Loss: 0.5680
Epoch [8/10], Loss: 0.5140
Epoch [9/10], Loss: 0.4940
Epoch [10/10], Loss: 0.4804


In [27]:
correct = 0
total = 0
model.eval()
with torch.no_grad():
    for inputs, labels in dataloader:
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Accuracy: {accuracy:.2f}%")


Accuracy: 80.00%
